In [1]:
import os
import cv2
import torch
from torch.utils.data import Dataset
import numpy as np

class VideoDataset(Dataset):

    def __init__(self, frame_path, seq_len=5):

        self.seq_len = seq_len
        self.samples = []

        folders = os.listdir(frame_path)

        for folder in folders:
            folder_path = os.path.join(frame_path, folder)

            frames = sorted(os.listdir(folder_path))

            for i in range(len(frames) - seq_len):
                seq = frames[i:i+seq_len+1]
                self.samples.append(
                    [os.path.join(folder_path, f) for f in seq]
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        paths = self.samples[idx]

        imgs = []

        for p in paths:
            img = cv2.imread(p)
            img = cv2.resize(img,(128,128))
            img = img/255.0
            imgs.append(img)

        imgs = np.array(imgs)

        input_seq = imgs[:-1]
        target = imgs[-1]

        return torch.tensor(input_seq).permute(0,3,1,2).float(), \
               torch.tensor(target).permute(2,0,1).float()

In [2]:
dataset = VideoDataset("../shanghaitech/training/frames")

print("Total training samples:", len(dataset))

Total training samples: 272865
